In [0]:
%sql
use catalog misgauravcatalog;
create database if not exists retaildb;
use retaildb; 

In [0]:
%sql
create table if not exists retaildb.orders_bronze
(
order_id int,
order_date string,
customer_id int,
order_status string,
filename string,
createdon timestamp
)
using delta
location "gs://databricksfolder/bronze/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_silver
(
order_id int,
order_date date,
customer_id int,
order_status string,
order_year int GENERATED ALWAYS AS (YEAR(order_date)),
order_month int GENERATED ALWAYS AS (MONTH(order_date)),
createdon timestamp,
modifiedon timestamp
)
using delta
location "gs://databricksfolder/silver/"
partitioned by (order_status)
TBLPROPERTIES(delta.enableChangeDataFeed = true);

In [0]:
%sql
create table if not exists retaildb.orders_gold
(
customer_id int,
order_status string,
order_year int,
num_orders int
)
using delta
location "gs://databricksfolder/gold/"
TBLPROPERTIES(delta.enableChangeDataFeed = true)

In [0]:
%sql
copy into retaildb.orders_bronze from (
  select order_id::int,
  order_date::string,
  customer_id::int,
  order_status::string,
  _metadata.file_path as filename,
  current_timestamp() as createdon
  from 'gs://databricksfolder/raw/'
)
fileformat = CSV
format_options('header'='true');


In [0]:
# ====================================================================
# 1. Ensure Silver target path and _delta_log are physically initialized
# ====================================================================
try:
    spark.sql("DESCRIBE HISTORY retaildb.orders_silver")
    silver_initialized = True
except Exception:
    silver_initialized = False

if not silver_initialized:
    print("First run detected! Force-initializing Silver directory on GCS...")
    from pyspark.sql.types import StructType, StructField, IntegerType, DateType, StringType, TimestampType
    from datetime import datetime

    schema = StructType([
        StructField("order_id", IntegerType(), True),
        StructField("order_date", DateType(), True),
        StructField("customer_id", IntegerType(), True),
        StructField("order_status", StringType(), True),
        StructField("order_year", IntegerType(), True),
        StructField("order_month", IntegerType(), True),
        StructField("createdon", TimestampType(), True),
        StructField("modifiedon", TimestampType(), True)
    ])
    dummy_data = [(-1, datetime.strptime('1970-01-01', '%Y-%m-%d').date(), -1, 'INITIALIZED', 1970, 1, datetime.now(), datetime.now())]
    dummy_df = spark.createDataFrame(dummy_data, schema)
    dummy_df.write.format("delta").mode("overwrite").option("mergeSchema", "true").save("gs://databricksfolder/silver/")
    spark.sql("DELETE FROM retaildb.orders_silver WHERE order_id = -1")

# ====================================================================
# 2. Smart Source Routing: Read directly on Version 0, use CDF on Version 1+
# ====================================================================
history_df = spark.sql("DESCRIBE HISTORY retaildb.orders_bronze")
latest_version = history_df.select("version").first()[0]

if latest_version == 0:
    # Day 1: Read your 4 rows directly from the table since Version 0 has no CDF side-logs
    print("Processing initial baseline batch directly from Bronze data files...")
    changes_df = spark.read.table("retaildb.orders_bronze") \
        .filter("order_id > 0 AND customer_id > 0 AND order_status IN ('CLOSED', 'PENDING_PAYMENT')")
else:
    # Day 2+: Safely read incrementally using Change Data Feed starting from Version 1
    print(f"Processing incremental changes via CDF (Starting at Version 1, Current Table Version: {latest_version})...")
    changes_df = spark.read.format("delta") \
        .option("readChangeData", "true") \
        .option("startingVersion", 1) \
        .load("gs://databricksfolder/bronze/") \
        .filter("order_id > 0 AND customer_id > 0 AND order_status IN ('CLOSED', 'PENDING_PAYMENT')")

# Expose the dataframe back to your SQL engine
changes_df.createOrReplaceTempView("orders_bronze_changes")

# ====================================================================
# 3. Run Production MERGE
# ====================================================================
print("Executing Silver tier MERGE upsert logic...")
spark.sql("""
    MERGE INTO retaildb.orders_silver tgt
    USING orders_bronze_changes src ON tgt.order_id = src.order_id
    WHEN MATCHED THEN
      UPDATE SET tgt.order_status = src.order_status, tgt.customer_id = src.customer_id, tgt.modifiedon = CURRENT_TIMESTAMP()
    WHEN NOT MATCHED THEN
      INSERT (order_id, order_date, customer_id, order_status, createdon, modifiedon) 
      VALUES (src.order_id, src.order_date, src.customer_id, src.order_status, CURRENT_TIMESTAMP(), CURRENT_TIMESTAMP())
""")
print("Silver pipeline processed completely and cleanly!")

In [0]:
# 1. Safely check if the physical Gold storage log folder exists on GCS
try:
    spark.sql("DESCRIBE HISTORY retaildb.orders_gold")
    gold_initialized = True
except Exception:
    gold_initialized = False

# 2. If it's missing, force-initialize the directory structure
if not gold_initialized:
    print("First run detected! Force-initializing Gold _delta_log directory on GCS...")
    
    # Calculate the initial aggregation dataframe directly from Silver
    initial_gold_df = spark.sql("""
        SELECT customer_id, order_status, order_year, COUNT(order_id) AS num_orders
        FROM retaildb.orders_silver 
        GROUP BY customer_id, order_status, order_year
    """)
    
    # Perform a raw path-based write to build the physical GCS log folder safely
    initial_gold_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .save("gs://databricksfolder/gold/")
    print("Gold physical log structure created successfully!")

# 3. Subsequent Runs / Production Daily Overwrite
print("Executing production Gold aggregation overwrite...")
spark.sql("""
    INSERT OVERWRITE TABLE retaildb.orders_gold
    SELECT customer_id, order_status, order_year, COUNT(order_id) AS num_orders
    FROM retaildb.orders_silver 
    GROUP BY customer_id, order_status, order_year
""")
print("Gold tier aggregation processed successfully!")